# 3.4 Dynamic Tools (adaptado a lab local)

**Qué se ve aquí:** `@wrap_model_call` filtra **qué tools ve el modelo** según el runtime context (`request.override(tools=...)`). Usuario `external` → solo `web_search`; `internal` → también `sql_query`. Es control de acceso: el modelo no puede llamar una tool que no ve.

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-3/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest")

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: Tavily sin límite devuelve mucho texto (latencia en CPU) → max_results=3.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.tools import tool
# from typing import Dict, Any
# from tavily import TavilyClient
# from langchain_community.utilities import SQLDatabase
#
# tavily_client = TavilyClient()
#
# db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")
#
#
# @tool
# def web_search(query: str) -> Dict[str, Any]:
#
#     """Search the web for information"""
#
#     return tavily_client.search(query)
#
# @tool
# def sql_query(query: str) -> str:
#
#     """Obtain information from the database using SQL queries"""
#
#     try:
#         return db.run(query)
#     except Exception as e:
#         return f"Error: {e}"

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: Tavily sin límite devuelve mucho texto (latencia en CPU) → max_results=3.
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query, max_results=3)   # menos tokens en CPU

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [ ]:
# Sin cambios
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [ ]:
# Sin cambios
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
#
# agent = create_agent(
#     model="gpt-5-nano",
#     tools=[web_search, sql_query],
#     middleware=[dynamic_tool_call],
#     context_schema=UserRole
# )

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" es API de pago.
from langchain.agents import create_agent

agent = create_agent(
    model=MODEL,
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [ ]:
# Sin cambios
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

### 🟢 EXTRA (no está en el curso): el mismo agente con rol `internal`
Con `internal` el middleware no filtra y el modelo ve también `sql_query`. Compara con la respuesta de `external`.

In [ ]:
# 🟢 ADAPTADO LOCAL — EXTRA: probar el rol interno
response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)